This dataset was manipulated from the data_jobs dataset created by Luke Barousse containing hundreds of thousands of real-world job postings related to data analytics, data engineering, and data science roles. It was manipulated to include only data analyst jobs using the following code:

```python
data_jobs = pd.read_csv("data_jobs.csv")

data_jobs = data_jobs[data_jobs['job_title'] == 'Data Analyst'].reset_index(drop=True)

data_jobs.to_csv('data_analyst_jobs.csv', index=False)
```

This dataset includes detailed information such as job titles, salaries, employment type, company location, remote status, required skills, and education requirements. The dataset supports analysis of hiring trends, skill demand, and salary patterns across the data industry.

In [1]:
import pandas as pd

no_degree_jobs = pd.read_csv("../data/data_analyst_jobs.csv")
no_degree_jobs.head()

,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills,job_type_skills
0,Data Analyst,Data Analyst,"Guadalajara, Jalisco, Mexico",via BeBee México,Full-time,False,Mexico,2023-01-14 13:18:07,False,False,Mexico,NaN,NaN,NaN,Hewlett Packard Enterprise,"['r', 'python', 'sql', 'nosql', 'power bi', 't...","{'analyst_tools': ['power bi', 'tableau'], 'pr..."
1,Data Analyst,Data Analyst,"Warsaw, Poland",via Praca Trabajo.org,Full-time,False,Poland,2023-10-16 13:36:54,False,False,Poland,NaN,NaN,NaN,Glovo,"['sql', 'python', 'r', 'redshift', 'pandas', '...","{'analyst_tools': ['excel', 'looker', 'tableau..."
2,Data Analyst,Data Analyst,"Des Moines, IA",via Trabajo.org,Full-time,False,"Illinois, United States",2023-11-06 13:01:22,False,True,United States,NaN,NaN,NaN,Assuredpartners,NaN,NaN
3,Data Analyst,Data Analyst,Singapore,via BeBee Singapore,Full-time,False,Singapore,2023-12-20 13:15:45,True,False,Singapore,NaN,NaN,NaN,Moovaz,['sql'],{'programming': ['sql']}
4,Data Analyst,Data Analyst,"Tampa, FL",via LinkedIn,Full-time,False,"Florida, United States",2023-01-19 13:19:45,False,False,United States,NaN,NaN,NaN,Citi,"['sql', 'python', 'unix', 'excel', 'jira']","{'analyst_tools': ['excel'], 'async': ['jira']..."


In [2]:
no_degree_jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 41950 entries, 0 to 41949
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   job_title_short        41950 non-null  str    
 1   job_title              41950 non-null  str    
 2   job_location           41861 non-null  str    
 3   job_via                41942 non-null  str    
 4   job_schedule_type      40825 non-null  str    
 5   job_work_from_home     41950 non-null  bool   
 6   search_location        41950 non-null  str    
 7   job_posted_date        41950 non-null  str    
 8   job_no_degree_mention  41950 non-null  bool   
 9   job_health_insurance   41950 non-null  bool   
 10  job_country            41950 non-null  str    
 11  salary_rate            2383 non-null   str    
 12  salary_year_avg        1309 non-null   float64
 13  salary_hour_avg        1051 non-null   float64
 14  company_name           41950 non-null  str    
 15  job_skills   

In [3]:
no_degree_jobs.isnull().sum()

job_title_short              0
job_title                    0
job_location                89
job_via                      8
job_schedule_type         1125
job_work_from_home           0
search_location              0
job_posted_date              0
job_no_degree_mention        0
job_health_insurance         0
job_country                  0
salary_rate              39567
salary_year_avg          40641
salary_hour_avg          40899
company_name                 0
job_skills                6449
job_type_skills           6449
dtype: int64

In [4]:
# Define the columns to keep
columns_to_keep = [
    'job_title', 'salary_year_avg', 'job_schedule_type', 'job_country',
    'job_work_from_home', 'search_location', 'job_skills', 'job_no_degree_mention', 'job_posted_date', 'company_name'
]

# Create new data_analyst_jobs with columns to keep
no_degree_jobs = no_degree_jobs[columns_to_keep]
no_degree_jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 41950 entries, 0 to 41949
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   job_title              41950 non-null  str    
 1   salary_year_avg        1309 non-null   float64
 2   job_schedule_type      40825 non-null  str    
 3   job_country            41950 non-null  str    
 4   job_work_from_home     41950 non-null  bool   
 5   search_location        41950 non-null  str    
 6   job_skills             35501 non-null  str    
 7   job_no_degree_mention  41950 non-null  bool   
 8   job_posted_date        41950 non-null  str    
 9   company_name           41950 non-null  str    
dtypes: bool(2), float64(1), str(7)
memory usage: 2.6 MB


In [5]:
# Renaming columns for consistency
no_degree_jobs = no_degree_jobs.rename(columns={
    'salary_year_avg': 'salary_usd',
    'job_schedule_type': 'employment_type',
    'job_country': 'company_location',
    'job_work_from_home': 'is_remote',
    'search_location': 'employee_location',
    'job_no_degree_mention': 'no_degree_mention',
    'job_posted_date': 'posting_date'
})

no_degree_jobs.columns

Index(['job_title', 'salary_usd', 'employment_type', 'company_location',
       'is_remote', 'employee_location', 'job_skills', 'no_degree_mention',
       'posting_date', 'company_name'],
      dtype='str')

## 🧹 Cleaning the `job_skills` Column

The `job_skills` column in the `no_degree_jobs` dataset contains multiple different formats depending on how the data was originally stored. Some rows contain:

- `NaN` values  
- real Python lists  
- NumPy arrays  
- stringified lists such as `"['sql','python']"`  
- comma‑separated strings such as `"sql, python, tableau"`  

To standardize this column for analysis, I created a `clean_skills()` function that converts **all** of these formats into a clean Python list of skill strings. This ensures the column is consistent and ready for downstream processing such as counting skills, exploding lists, or inserting into a database.

The function handles each possible case:

- **NaN values** → converted to empty lists  
- **real Python lists** → returned unchanged  
- **NumPy arrays** → converted to lists  
- **stringified lists** → parsed into real lists  
- **comma‑separated strings** → split into lists  
- **anything else** → returned as an empty list  

### Cleaning Function

```python
def clean_skills(x):
    # Case 1: NaN (float)
    if isinstance(x, float):
        return []
    
    # Case 2: Real Python list
    if isinstance(x, list):
        return x
    
    # Case 3: NumPy array → convert to list
    if hasattr(x, "tolist"):
        return [skill.strip() for skill in x.tolist()]
    
    # Case 4: Stringified list like "['sql','python']"
    if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        # Remove brackets
        inner = x[1:-1]
        # Split by comma
        items = inner.split(',')
        # Strip quotes and whitespace
        return [skill.strip().strip("'").strip('"') for skill in items]
    
    # Case 5: Comma-separated string
    if isinstance(x, str):
        return [skill.strip() for skill in x.split(',')]
    
    # Fallback
    return []
```

This function is applied to the `job_skills` column to ensure every row contains a clean, standardized list of skills.

In [6]:
def clean_skills(x):
    # Case 1: NaN (float)
    if isinstance(x, float):
        return []
    
    # Case 2: Real Python list
    if isinstance(x, list):
        return x
    
    # Case 3: NumPy array → convert to list
    if hasattr(x, "tolist"):
        return [skill.strip() for skill in x.tolist()]
    
    # Case 4: Stringified list like "['sql','python']"
    if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        # Remove brackets
        inner = x[1:-1]
        # Split by comma
        items = inner.split(',')
        # Strip quotes and whitespace
        return [skill.strip().strip("'").strip('"') for skill in items]
    
    # Case 5: Comma-separated string
    if isinstance(x, str):
        return [skill.strip() for skill in x.split(',')]
    
    # Fallback
    return []

In [7]:
no_degree_jobs['job_skills'] = no_degree_jobs['job_skills'].apply(clean_skills)

In [8]:
no_degree_jobs['job_skills'].head()

0           [r, python, sql, nosql, power bi, tableau]
1    [sql, python, r, redshift, pandas, excel, look...
2                                                   []
3                                                [sql]
4                     [sql, python, unix, excel, jira]
Name: job_skills, dtype: object

In [9]:
# Create indicator columns for each skill of interest
skills_to_track = ['python', 'sql', 'tableau']

for skill in skills_to_track:
    no_degree_jobs[f"has_{skill}"] = no_degree_jobs['job_skills'].apply(
        lambda skills: skill in skills
    )

# Count skill frequency for python, sql, and tableau
no_degree_jobs[['has_python', 'has_sql', 'has_tableau']].sum()

has_python     14212
has_sql        24036
has_tableau    12084
dtype: int64

In [10]:
# Count individual skills
skills_exploded = no_degree_jobs['job_skills'].explode()
skills_exploded.value_counts()

job_skills
sql         24036
excel       15119
python      14212
tableau     12084
power bi    10127
            ...  
pascal          1
mxnet           1
chainer         1
wsl             1
twilio          1
Name: count, Length: 211, dtype: int64

In [11]:
no_degree_jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 41950 entries, 0 to 41949
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   job_title          41950 non-null  str    
 1   salary_usd         1309 non-null   float64
 2   employment_type    40825 non-null  str    
 3   company_location   41950 non-null  str    
 4   is_remote          41950 non-null  bool   
 5   employee_location  41950 non-null  str    
 6   job_skills         41950 non-null  object 
 7   no_degree_mention  41950 non-null  bool   
 8   posting_date       41950 non-null  str    
 9   company_name       41950 non-null  str    
 10  has_python         41950 non-null  bool   
 11  has_sql            41950 non-null  bool   
 12  has_tableau        41950 non-null  bool   
dtypes: bool(5), float64(1), object(1), str(6)
memory usage: 2.8+ MB


In [12]:
no_degree_jobs['company_name'].value_counts()

company_name
Robert Half                            385
Insight Global                         316
Emprego                                224
Peroptyx                               200
Confidenziale                          139
                                      ... 
Trip.com Group                           1
Sutrix Solutions                         1
HelloConnect                             1
C-Care (Mauritius) Ltd                   1
GUS Global Services India Pvt. Ltd.      1
Name: count, Length: 18939, dtype: int64

In [13]:
no_degree_jobs['no_degree_mention'].value_counts()

no_degree_mention
False    23415
True     18535
Name: count, dtype: int64

In [14]:
no_degree_jobs = no_degree_jobs[
    no_degree_jobs['no_degree_mention'] == True
].copy()

In [15]:
no_degree_jobs['degree_flag'] = 0
no_degree_jobs.info()

<class 'pandas.DataFrame'>
Index: 18535 entries, 3 to 41949
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   job_title          18535 non-null  str    
 1   salary_usd         345 non-null    float64
 2   employment_type    18003 non-null  str    
 3   company_location   18535 non-null  str    
 4   is_remote          18535 non-null  bool   
 5   employee_location  18535 non-null  str    
 6   job_skills         18535 non-null  object 
 7   no_degree_mention  18535 non-null  bool   
 8   posting_date       18535 non-null  str    
 9   company_name       18535 non-null  str    
 10  has_python         18535 non-null  bool   
 11  has_sql            18535 non-null  bool   
 12  has_tableau        18535 non-null  bool   
 13  degree_flag        18535 non-null  int64  
dtypes: bool(5), float64(1), int64(1), object(1), str(6)
memory usage: 1.5+ MB


## 🧹 Handling Salary Conversion with NaNs

In Notebook 01, the `salary_usd` column contained **no missing values**, so converting salary values from floats to integers was straightforward:

```python
data_analyst_jobs['salary_usd'] = data_analyst_jobs['salary_usd'].astype(int)
```

However, in Notebook 02, the `no_degree_jobs` dataset contains only **345 salary values** and more than **18,000 missing values (NaNs)**. Standard integer types cannot store `NaN`, so attempting to cast the column directly to `int` or even Pandas’ nullable `Int64` type results in a `TypeError`:

```
TypeError: Cannot cast array data from dtype('float64') to dtype('int64') according to the rule 'safe'
```

To safely convert the valid salary floats to integers **while preserving NaNs**, I used Pandas’ nullable integer type. This allows the column to contain both integers and missing values:

```python
no_degree_jobs['salary_usd'] = (
    no_degree_jobs['salary_usd']
    .round()
    .astype('Int64')
)
```

This approach:

- converts valid salary floats (e.g., `117500.0`) into integers (`117500`)
- keeps `NaN` values intact
- avoids casting errors caused by non‑nullable integer types
- ensures consistent formatting with Notebook 01 while accommodating the missing salary data present in Notebook 02

In [16]:
no_degree_jobs[no_degree_jobs['salary_usd'].notna()]['salary_usd'].head(10)

157     117500.0
468     100500.0
887      90000.0
968      75000.0
999      55000.0
1343     60000.0
1895     57500.0
1999    100000.0
2722     75000.0
3338    125000.0
Name: salary_usd, dtype: float64

In [17]:
no_degree_jobs['salary_usd'] = (
    no_degree_jobs['salary_usd']
    .round()
    .astype('Int64')
)

no_degree_jobs.loc[no_degree_jobs['salary_usd'].notna(), 'salary_usd'].head(10)


157     117500
468     100500
887      90000
968      75000
999      55000
1343     60000
1895     57500
1999    100000
2722     75000
3338    125000
Name: salary_usd, dtype: Int64

In [18]:
# Function to put salary into tiers for better visualizations later
def categorize_salary(df: pd.DataFrame, salary_column: str) -> pd.DataFrame:
    """
    Categorizes salaries into income tiers: 'Low', 'Mid', or 'High' based on defined thresholds.

    Args:
        df (pd.DataFrame): The DataFrame containing a salary column as integers.
        salary_column (str): The name of the column with salary values (e.g., 'salary_usd').

    Returns:
        pd.DataFrame: The modified DataFrame with a new 'salary_tier' column.
    """
    bins = [0, 50000, 100000, float('inf')]
    labels = ['Low', 'Mid', 'High']
    df['salary_tier'] = pd.cut(df[salary_column], bins=bins, labels=labels, include_lowest=True)
    return df

In [19]:
no_degree_jobs = categorize_salary(no_degree_jobs, 'salary_usd')

In [20]:
no_degree_jobs.columns

Index(['job_title', 'salary_usd', 'employment_type', 'company_location',
       'is_remote', 'employee_location', 'job_skills', 'no_degree_mention',
       'posting_date', 'company_name', 'has_python', 'has_sql', 'has_tableau',
       'degree_flag', 'salary_tier'],
      dtype='str')

In [21]:
no_degree_jobs.loc[no_degree_jobs['salary_usd'].notna(), 'salary_usd'].head(10)


157     117500
468     100500
887      90000
968      75000
999      55000
1343     60000
1895     57500
1999    100000
2722     75000
3338    125000
Name: salary_usd, dtype: Int64

In [22]:
no_degree_jobs.loc[
    no_degree_jobs['salary_usd'].notna(),
    ['salary_usd', 'salary_tier']
].head()

,salary_usd,salary_tier
157,117500,High
468,100500,High
887,90000,Mid
968,75000,Mid
999,55000,Mid


In [23]:
no_degree_jobs['is_remote'].value_counts()

is_remote
False    17347
True      1188
Name: count, dtype: int64

In [24]:
no_degree_jobs['employee_location'].value_counts()

employee_location
United Kingdom               2513
New York, United States      1154
California, United States     914
Texas, United States          862
Georgia                       716
                             ... 
Bolivia                         1
Rwanda                          1
Palestine                       1
Malawi                          1
Nepal                           1
Name: count, Length: 127, dtype: int64

In [25]:
no_degree_jobs['employment_type'].value_counts()

employment_type
Full-time                               15827
Contractor                               1397
Part-time                                 261
Full-time and Part-time                   110
Temp work                                  96
Full-time and Temp work                    90
Full-time and Contractor                   76
Contractor and Temp work                   73
Full-time, Part-time, and Contractor       31
Internship                                 13
Part-time and Contractor                   11
Full-time, Contractor, and Temp work        7
Part-time and Temp work                     6
Full-time, Part-time, and Temp work         3
Full-time and Per diem                      1
Volunteer                                   1
Name: count, dtype: int64

### **Normalizing Employment Type**

The `employment_type` column in the `no_degree_jobs` dataset contains long descriptive strings and multi‑category combinations such as “Full-time and Part-time” or “Contractor and Temp work.” In contrast, the `degree_jobs` dataset used in Notebook 01 already contains four standardized codes:

- **FT** — Full‑time  
- **PT** — Part‑time  
- **CT** — Contractor  
- **FL** — Freelance  

To ensure both datasets can be joined, compared, and visualized consistently, I normalized the employment types in Notebook 02 to match the four‑code system used in Notebook 01. The mapping function below converts all descriptive and multi‑category strings into the same standardized codes:

```python
def map_employment_type(x):
    x = str(x).lower()

    if 'full-time' in x:
        return 'FT'
    if 'part-time' in x:
        return 'PT'
    if 'contractor' in x:
        return 'CT'
    if 'temp' in x or 'per diem' in x:
        return 'CT'
    if 'intern' in x:
        return 'PT'
    if 'volunteer' in x:
        return 'PT'

    return None
```

This produces a clean `employment_type_clean` column that aligns with Notebook 01 and ensures cohesive joins and visualizations across both datasets.

In [26]:
def map_employment_type(x):
    x = str(x).lower()

    if 'full-time' in x:
        return 'FT'
    if 'part-time' in x:
        return 'PT'
    if 'contractor' in x:
        return 'CT'
    if 'temp' in x or 'per diem' in x:
        return 'CT'   # temp work behaves like contractor in your dataset
    if 'intern' in x:
        return 'PT'   # internships behave like part-time
    if 'volunteer' in x:
        return 'PT'   # volunteers behave like part-time

    return None


In [27]:
no_degree_jobs['employment_type_clean'] = no_degree_jobs['employment_type'].apply(map_employment_type)
no_degree_jobs['employment_type_clean'].value_counts()

employment_type_clean
FT    16145
CT     1566
PT      292
Name: count, dtype: int64

In [28]:
# Renamed column after cleaning
no_degree_jobs = no_degree_jobs.drop(columns='employment_type')
no_degree_jobs = no_degree_jobs.rename(
    columns={'employment_type_clean': 'employment_type'}
)

In [29]:
no_degree_jobs['posting_date'].head()

3     2023-12-20 13:15:45
8     2023-10-11 13:17:59
9     2023-09-06 13:08:51
11    2023-12-28 13:37:04
12    2023-04-13 13:32:55
Name: posting_date, dtype: str

In [30]:
# Convert to datetime
no_degree_jobs['posting_date'] = pd.to_datetime(no_degree_jobs['posting_date'])

# Keep only the date part
no_degree_jobs['posting_date'] = no_degree_jobs['posting_date'].dt.date

no_degree_jobs['posting_date'].head()

3     2023-12-20
8     2023-10-11
9     2023-09-06
11    2023-12-28
12    2023-04-13
Name: posting_date, dtype: object

In [31]:
# Create index and job id column
no_degree_jobs = no_degree_jobs.reset_index().rename(columns={'index': 'job_id'})

In [32]:
no_degree_jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 18535 entries, 0 to 18534
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   job_id             18535 non-null  int64   
 1   job_title          18535 non-null  str     
 2   salary_usd         345 non-null    Int64   
 3   company_location   18535 non-null  str     
 4   is_remote          18535 non-null  bool    
 5   employee_location  18535 non-null  str     
 6   job_skills         18535 non-null  object  
 7   no_degree_mention  18535 non-null  bool    
 8   posting_date       18535 non-null  object  
 9   company_name       18535 non-null  str     
 10  has_python         18535 non-null  bool    
 11  has_sql            18535 non-null  bool    
 12  has_tableau        18535 non-null  bool    
 13  degree_flag        18535 non-null  int64   
 14  salary_tier        345 non-null    category
 15  employment_type    18003 non-null  str     
dtypes: Int64(1), bo

## 🧹 Handling Missing Values

After completing the core cleaning steps—standardizing job skills, converting salary values, normalizing employment types, and creating the `job_id` and `degree_flag` columns—I evaluated missing values to ensure the final dataset remains reliable and ready for SQL loading and analysis.

```python
no_degree_jobs.isna().sum()
```

### `salary_usd`

Many job postings do not report salary information, so missing salary values are expected. Consistent with Notebook 01, I retained these values rather than filling them with estimates or dropping otherwise valid postings.

The `salary_usd` column uses Pandas' nullable `Int64` type, which preserves valid salary values as integers while allowing missing values (`pd.NA`). Retaining missing salary values avoids introducing artificial estimates and preserves the original information available in each job posting.

### `salary_tier`

`salary_tier` is derived from `salary_usd` using defined salary ranges (`Low`, `Mid`, and `High`). Therefore, when a job posting does not include a salary, no salary tier can be assigned.

Missing values in `salary_tier` are retained because they accurately reflect missing salary information in the original posting. No tier is imputed, since assigning one without a reported salary would introduce an unsupported assumption.

For salary-tier analysis, only records where `salary_tier` is not missing will be included.

### `employment_type`

Some records do not provide an employment type in the original source data, so their standardized `employment_type` value remains missing. These values are retained because the original posting does not provide enough information to classify the employment arrangement accurately.

The mapping function standardizes available employment types into codes aligned with Notebook 01:

- `FT` — Full-time
- `PT` — Part-time
- `CT` — Contractor
- `FL` — Freelance

FL is retained as part of the standardized schema used across the project, although no freelance records are present in this dataset. It also maps temporary work and per-diem roles to `CT`, and internships and volunteer roles to `PT`, so the final categories remain consistent for later analysis.

### `job_skills`

The original `job_skills` column included missing values. The `clean_skills()` function converted these values to empty lists (`[]`), meaning the field no longer contains null values. An empty list indicates that no skills were provided in the original posting; it does not imply that the job requires no skills.

### Other columns

The remaining retained columns, including `company_name`, `is_remote`, `no_degree_mention`, and `degree_flag`, contain no missing values. `posting_date` was converted to a date-only format after validation.

### Approach

To preserve data integrity and remain consistent with Notebook 01:

- Missing values are not replaced with guesses, placeholders, or statistical estimates.
- Rows are not dropped solely because a source field is missing.
- Missing values are retained when they represent information not provided in the original job posting.
- Analyses that require salary or employment-type information will filter to non-missing records for the relevant calculation.

## Duplicate Assessment

Exact duplicates were evaluated across all meaningful analytical fields while excluding job_id, which serves as the generated unique identifier for each record.

Because job_skills is stored as a list and lists are not hashable, a temporary tuple representation of the column was created so that skills could be included in the duplicate comparison.

The duplicate check identified 241 repeated records that would be removed while retaining the first occurrence of each exact duplicate group. Inspection of the duplicate records confirmed that the identified records contained identical substantive job information, with the generated job_id distinguishing the repeated records.

The repeated records were therefore removed while retaining the first occurrence of each duplicate group. The temporary skills-tuple column was then removed, and the dataset was rechecked to confirm that no exact duplicates remained.

In [33]:
# Create temporary hashable version of job_skills
no_degree_jobs['job_skills_tuple'] = (
    no_degree_jobs['job_skills'].apply(tuple)
)

# Compare all meaningful analytical columns except the generated ID
duplicate_columns = no_degree_jobs.columns.drop(
    ['job_id', 'job_skills']
)

# Count exact duplicates
duplicate_count = no_degree_jobs.duplicated(
    subset=duplicate_columns
).sum()

duplicate_count

np.int64(241)

In [34]:
duplicates = no_degree_jobs[
    no_degree_jobs.duplicated(
        subset=duplicate_columns,
        keep=False
    )
].sort_values(
    by=['company_name', 'job_title', 'posting_date']
)

duplicates

,job_id,job_title,salary_usd,company_location,is_remote,employee_location,job_skills,no_degree_mention,posting_date,company_name,has_python,has_sql,has_tableau,degree_flag,salary_tier,employment_type,job_skills_tuple
9765,21730,Data Analyst,<NA>,Poland,False,Poland,"[sql, python, r, power bi, excel, tableau, pow...",True,2023-03-03,3Shape,True,True,True,0,NaN,FT,"(sql, python, r, power bi, excel, tableau, pow..."
10781,24027,Data Analyst,<NA>,Poland,False,Poland,"[sql, python, r, power bi, excel, tableau, pow...",True,2023-03-03,3Shape,True,True,True,0,NaN,FT,"(sql, python, r, power bi, excel, tableau, pow..."
12690,28428,Data Analyst,<NA>,Portugal,False,Portugal,[],True,2023-08-28,7Skin Lda,False,False,False,0,NaN,FT,()
12803,28656,Data Analyst,<NA>,Portugal,False,Portugal,[],True,2023-08-28,7Skin Lda,False,False,False,0,NaN,FT,()
12579,28202,Data Analyst,<NA>,United Kingdom,False,United Kingdom,"[spark, power bi, excel]",True,2023-02-10,ACS Recruitment,False,False,False,0,NaN,FT,"(spark, power bi, excel)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17725,40034,Data Analyst,<NA>,Thailand,False,Thailand,[],True,2023-11-24,บริษัท ซินด้า (ประเทศไทย) จำกัด,False,False,False,0,NaN,FT,()
17391,39228,Data Analyst,<NA>,Thailand,False,Thailand,[],True,2023-01-24,บริษัท สมาร์ทแมทโปร จำกัด,False,False,False,0,NaN,FT,()
18444,41740,Data Analyst,<NA>,Thailand,False,Thailand,[],True,2023-01-24,บริษัท สมาร์ทแมทโปร จำกัด,False,False,False,0,NaN,FT,()
7992,17745,Data Analyst,<NA>,Thailand,False,Thailand,[excel],True,2023-01-14,บริษัท เวสเทิร์น เดคอร์ คอร์ปอเรชั่น จำกัด (มห...,False,False,False,0,NaN,FT,"(excel,)"


In [35]:
# Remove duplicate records
no_degree_jobs = no_degree_jobs.drop_duplicates(
    subset=duplicate_columns,
    keep='first'
).copy()

In [36]:
# Remove temporary tuple column
no_degree_jobs = no_degree_jobs.drop(
    columns='job_skills_tuple'
)

In [37]:
# Recreate temporary tuple column for validation
no_degree_jobs['job_skills_tuple'] = (
    no_degree_jobs['job_skills'].apply(tuple)
)

duplicate_columns = no_degree_jobs.columns.drop(
    ['job_id', 'job_skills']
)

no_degree_jobs.duplicated(
    subset=duplicate_columns
).sum()

np.int64(0)

In [38]:
# Remove temporary tuple column
no_degree_jobs = no_degree_jobs.drop(
    columns='job_skills_tuple'
)

In [39]:
no_degree_jobs.shape

(18294, 16)

In [40]:
no_degree_jobs['job_id'].is_unique

True

In [41]:
no_degree_jobs.isna().sum()

job_id                   0
job_title                0
salary_usd           17949
company_location         0
is_remote                0
employee_location        0
job_skills               0
no_degree_mention        0
posting_date             0
company_name             0
has_python               0
has_sql                  0
has_tableau              0
degree_flag              0
salary_tier          17949
employment_type        529
dtype: int64

In [42]:
no_degree_jobs.info()

<class 'pandas.DataFrame'>
Index: 18294 entries, 0 to 18534
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   job_id             18294 non-null  int64   
 1   job_title          18294 non-null  str     
 2   salary_usd         345 non-null    Int64   
 3   company_location   18294 non-null  str     
 4   is_remote          18294 non-null  bool    
 5   employee_location  18294 non-null  str     
 6   job_skills         18294 non-null  object  
 7   no_degree_mention  18294 non-null  bool    
 8   posting_date       18294 non-null  object  
 9   company_name       18294 non-null  str     
 10  has_python         18294 non-null  bool    
 11  has_sql            18294 non-null  bool    
 12  has_tableau        18294 non-null  bool    
 13  degree_flag        18294 non-null  int64   
 14  salary_tier        345 non-null    category
 15  employment_type    17765 non-null  str     
dtypes: Int64(1), bool(5)